# Fase 4: Failure Classification (Dokter Diagnosa)

## Tujuan
Klasifikasi status mesin (Healthy / Warning / Critical) menggunakan Supervised Learning (Random Forest) dengan penanganan data imbalanced.

## Langkah-langkah
- 4.1: Rekayasa Target Label 3 Kelas (Healthy, Warning, Critical) via shift(-n)
- 4.2: Train/Test Split Linear (TimeSeriesSplit, TANPA shuffle)
- 4.3: Penerapan SMOTE pada Data Training saja
- 4.4: Training Random Forest Classifier
- 4.5: Evaluasi (Fokus Recall, bukan Accuracy)
- 4.6: Feature Importance Extraction
- 4.7: Ekspor Model (classifier_rf.pkl)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Melakukan Rekayasa Label Target dengan Teknik Vectorization (Fase 4)...\n")

# 1. Memuat data hasil ekstraksi Fase 2
df = pd.read_csv('../data/processed/sensor_features_engineered.csv', index_col='timestamp', parse_dates=True)

# 2. Membuat kolom bantuan 'next_failure' yang HANYA berisi index waktu saat failure == 1
# (Memanfaatkan Pandas Series Masking untuk mencegat bottleneck perubahan tipe object)
df['next_failure'] = df.index.to_series().where(df['failure'] == 1)

# 3. Terapkan Backward Fill spesifik per mesin menggunakan groupby
# Ini berfungsi menarik tanggal 'ledakan' di masa depan ke baris-baris masa kini
df['next_failure'] = df.groupby('machine_id')['next_failure'].bfill()

# 4. Hitung jarak persis dalam jam menggunakan selisih batas akhir dikurangi waktu saat ini
df['hours_to_failure'] = (df['next_failure'] - df.index).dt.total_seconds() / 3600.0

# 5. Terapkan Aturan Fungsi Label Kategorik melalui Vectorized np.select
# Critical (2): <= 48 Jam (2 Hari) sebelum rusak
# Warning (1): <= 168 Jam (7 Hari) dan > 48 Jam sebelum rusak
# Healthy (0): Selebihnya (> 168 Jam atau mesin tidak pernah rusak / NaN)
conditions = [
    (df['hours_to_failure'].notna()) & (df['hours_to_failure'] <= 48),
    (df['hours_to_failure'].notna()) & (df['hours_to_failure'] > 48) & (df['hours_to_failure'] <= 168)
]
choices = [2, 1]
df['status_label'] = np.select(conditions, choices, default=0)

# 6. Membersihkan DataFrame (Buang jejak rekayasa dan nilai kosong)
df.drop(columns=['failure', 'next_failure', 'hours_to_failure'], inplace=True, errors='ignore')
df.dropna(inplace=True)

# 7. Mencetak Value Counts (Distribusi 3 Kelas Kita) 
print("--- Distribusi Status Label ('Dokter Diagnosa') ---")
print("0 = Healthy (Mesin Normal / > 7 Hari menuju meledak)")
print("1 = Warning (Gejala Dini <= 7 Hari menuju meledak)")
print("2 = Critical (Darurat <= 48 Jam menuju meledak)\n")
display(df['status_label'].value_counts())

print("\n✅ Kecepatan Tinggi! Rekayasa Target Kelas (Langkah 4.1) Selesai!")

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# --- 4.2 Train/Test Split Linear (Time-Series) ---
print("Memulai Splitting Data & SMOTE (Fase 4)...\n")

# 1. Pisahkan fitur (X) dan label (y)
# Sebagai kehati-hatian, buang juga machine_id karena ia teks dan bukan fitur numerik perhitungan
X = df.drop(columns=['status_label', 'machine_id'])
y = df['status_label']

# 2 & 3. Pembagian Data (Train/Test Split)
# HUKUM MUTLAK RUNUT WAKTU: shuffle=False agar masa lalu tidak bocor ke ujian masa depan!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)
print(f"Distribusi Pembagian Asli -> Train: {X_train.shape[0]} baris, Test: {X_test.shape[0]} baris\n")

# --- 4.3 Penerapan SMOTE ---
# 4 & 5. Menghasilkan Sintetis Kelas Minor (SMOTE) HANYA pada Data Latih
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 6. Cetak perbandingan hasil penyeimbangan alam semesta matriks kita
print("--- Distribusi Kelas Latih (SEBELUM SMOTE) ---")
display(y_train.value_counts())

print("\n--- Distribusi Kelas Latih (SESUDAH SMOTE) ---")
display(y_train_smote.value_counts())

print("\n✅ Split Data & SMOTE (Langkah 4.2 & 4.3) Selesai! Data siap masuk algoritma Classifier.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# --- 4.4 Training Random Forest Classifier & Feature Importance ---
print("Memulai pelatihan model Supervised: Random Forest Classifier (Otomasi Dokter Mesin)...")
print("Status: Memproses ratusan ribu baris data (SMOTE). Ini akan memakan waktu kurang lebih 1-2 menit...\n")

# 1 & 2. Inisiasi Model (n_jobs=-1 berarti mengerahkan seluruh silinder core Prosesor PC/Server)
model_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)

# 3. Latih Modelnya! HANYA KEPADA DATA SMOTE! 
model_rf.fit(X_train_smote, y_train_smote)

# 4. Prediksikan Tes Ujian dengan data murni
y_pred = model_rf.predict(X_test)
print("✅ Pelatihan Random Forest Selesai!\n")


# 5. Ekstraksi Otak AI: Feature Importances
print("--- Analisis Anatomi Keputusan AI (Feature Importance Top-10) ---")
# 6. Gabungkan list kolom asli dan nilai bobot AI ke DataFrame rapi
importances = model_rf.feature_importances_
feature_importances_df = pd.DataFrame({'Fitur': X.columns, 'Bobot_Persentase': importances})

# 7. Urutkan secara Descending dan tampilkan top 10
feature_importances_df = feature_importances_df.sort_values(by='Bobot_Persentase', ascending=False)
display(feature_importances_df.head(10))

print("\n(Interpretasi: Semakin tinggi persentasenya, semakin sering sensor ini dipakai AI untuk memvonis bahwa mesin akan meledak!)")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# --- 4.5 Evaluasi (Fokus Recall) ---
print("--- Confusion Matrix ---\n")

# Menampilkan crosstab yang dirapikan dengan Pandas
display(pd.crosstab(y_test, y_pred, rownames=['Asli'], colnames=['Prediksi']))

print("\n--- Classification Report ---\n")
# Classification report untuk metrik presisi, recall, dan f1-score
print(classification_report(y_test, y_pred))

In [ ]:
# --- 4.5.b Optimasi Evaluasi Ekstrem (Paranoia Threshold Tuning) ---
print("Melakukan Paranoia Threshold Tuning (0.15) pada model orisinal untuk memaksa ekstrasi Recall Critical...\n")

# 1. Ambil array probabilitas mentah (prediksi kepercayaan diri model per baris per kelas)
y_proba = model_rf.predict_proba(X_test)

# 2. Buat array tebakan awal (Hanya menandingkan probabilitas kelas 0 vs kelas 1)
y_pred_custom = np.argmax(y_proba[:, :2], axis=1)

# 3. Terapkan Aturan Paranoia: Timpa menjadi kelas 2 BILA peluang kritisnya menyentuh >= 15% (0.15)
y_pred_custom[y_proba[:, 2] >= 0.15] = 2

# 4. Cetak header evaluasi yang baru
print("\n--- Classification Report (Threshold 0.15) ---\n")

# 5. Cetak classification report dari tebakan yang sudah di-tuning ekstrim
print(classification_report(y_test, y_pred_custom))

print("✅ Interpretasi Threshold: Jika Recall '2' sudah mencapai target mulia Anda (>95%), model ini siap diekspor!")

In [ ]:
import os
import joblib

# --- 4.7 Ekspor Model (Deployment Preparation) ---
print("Mempersiapkan penobatan Dokter Diagnosa (Random Forest)...\n")

# 1 & 2. Siapkan wadah persemayaman di root folder proyek
model_dir = '../models/'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'classifier_rf.pkl')

# 3. Bekukan model_rf (orisinal) ke dalam biner .pkl via Joblib
# PERHATIAN: Kita mengawetkan obyek 'model_rf' yang disukai karena dialah
# yang menghasilkan performa Recall superior via Threshold Tuning paska-prediksi.
joblib.dump(model_rf, model_path)

# 4. Cetak Sabda Keberhasilan
print(f"✅ Otak Dokter Diagnosa (Random Forest) berhasil diekspor ke {model_path}!")
print("Insinyur Backend kita sekarang dapat mengaitkannya ke dalam API web prediksi nyata!")